<a href="https://colab.research.google.com/github/Pinkraaaa/Intro-to-AI-Group-9/blob/isaac-CNN/Group_9_CNN_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!git clone https://github.com/Pinkraaaa/Intro-to-AI-Group-9.git
%cd Intro-to-AI-Group-9

Cloning into 'Intro-to-AI-Group-9'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 33 (delta 9), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (33/33), 61.90 MiB | 21.76 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/Intro-to-AI-Group-9/Intro-to-AI-Group-9


In [6]:
!unzip -q Potato_Test.zip
!unzip -q Potato_Train.zip
!unzip -q Potato_Validate.zip

In [7]:
import torch
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import torch.optim as optim

In [8]:
#Make sure to match the directories after Pinkra makes the repo public

train_dir = "Train"
val_dir   = "Valid"
test_dir  = "Test"

In [9]:
#Resizing block

#Must be re-checked again
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.RandomResizedCrop(128, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

eval_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])




In [10]:
#Loading the dataset and dataloaders

train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset   = datasets.ImageFolder(val_dir, transform=eval_transform)
test_dataset  = datasets.ImageFolder(test_dir, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [11]:
#checking if it works
images, labels = next(iter(train_loader))
print(images.shape)          # expect: torch.Size([32, 3, 128, 128])
print(labels)
print(train_dataset.classes) # confirms class-to-index mapping

torch.Size([32, 3, 128, 128])
tensor([0, 0, 2, 1, 0, 0, 0, 0, 2, 0, 1, 1, 0, 2, 0, 2, 0, 1, 2, 0, 2, 0, 1, 2,
        0, 1, 1, 1, 0, 2, 1, 0])
['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy']


In [12]:
import torch
import torch.nn as nn


class FINALCNN(nn.Module):

    def __init__(self, num_classes=2):
        super().__init__()

        # Block 1
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2)
        )

        # Block 2
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2)
        )

        # Block 3
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2)
        )

        # Block 4
        self.conv4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )

        # Global Average Pooling
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Linear(256, 128),
            nn.ReLU(inplace=True),

            nn.Dropout(0.5),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)

        x = self.global_pool(x)

        x = self.classifier(x)

        return x

In [13]:
num_classes = len(train_dataset.classes)

model = FINALCNN(num_classes=num_classes)
print(model)

FINALCNN(
  (conv1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
 

In [14]:
# Loss function
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [15]:
print(train_dataset.classes)
print(len(train_dataset.classes))

['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy']
3


In [ ]:
# -------------------------
# Training
# -------------------------

best_val_accuracy = 0.0
epochs = 10

# Keep track of how long validation accuracy has not improved
patience = 5
patience_counter = 0

for epoch in range(epochs):

    # =========================
    # TRAINING
    # =========================

    model.train()

    running_train_loss = 0.0
    correct_train = 0
    total_train = 0

    print(f"\nEpoch {epoch + 1}/{epochs}")

    for batch_idx, (images, labels) in enumerate(train_loader):

        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # -------------------------
        # Track training statistics
        # -------------------------

        running_train_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    # Average training loss
    train_loss = running_train_loss / total_train

    # Training accuracy
    train_accuracy = 100 * correct_train / total_train


Epoch 1/10

Epoch 2/10


In [ ]:
 # =========================
    # VALIDATION
    # =========================
model.eval()

running_val_loss = 0.0
correct_val = 0
total_val = 0

with torch.no_grad():
  for images, labels in val_loader:
    outputs = model(images)
    loss = criterion(outputs, labels)

    running_val_loss += loss.item() * images.size(0)

    _, predicted = torch.max(outputs, 1)

    total_val += labels.size(0)
    correct_val += (predicted == labels).sum().item()



    # Average validation loss
    val_loss = running_val_loss / total_val

    # Validation accuracy
    val_accuracy = 100 * correct_val / total_val


    # =========================
    # PRINT RESULTS
    # =========================

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_accuracy:.2f}%")

    print(f"Val Loss: {val_loss:.4f}")
    print(f"Val Accuracy: {val_accuracy:.2f}%")